# Phase 1 Lab — Reference Solution

**Phase:** Python Foundation  
**Scenario:** A training organization receives learner records from JSON and CSV. Some rows are incomplete, duplicated, or invalid.

**Deliverable:** A small Python package-style processor that validates records, calculates per-learner summaries, logs rejected records, writes a clean CSV, and includes tests.

Use this only after completing your own attempt. Compare design decisions—not merely output.

## Requirements

        1. Define a `LearnerRecord` data class with validation rules.
2. Load records using `pathlib` and a context manager.
3. Normalize names and score types without silently accepting impossible scores.
4. Deduplicate by learner ID with an explicit policy.
5. Calculate count, mean, minimum, maximum, and pass status.
6. Write accepted and rejected outputs separately.
7. Add unit tests for normal, boundary, empty, and invalid cases.

        ## Acceptance criteria

        - The notebook runs from a clean kernel in order.
        - Inputs and outputs have explicit contracts.
        - Invalid, missing, extreme, duplicate, and unseen cases are considered.
        - Important invariants use assertions or tests.
        - Results include interpretation and limitations.
        - Generated artifacts are written under the course `artifacts/` folder.

## Planning worksheet

Complete before coding:

| Question | Your answer |
|---|---|
| What decision or user does the result serve? | |
| What does one row/object/event represent? | |
| What are the required inputs and types? | |
| What outputs and side effects are allowed? | |
| Which assumptions are most fragile? | |
| What is the simplest valid baseline? | |
| Which edge cases must be tested? | |
| How will you know the result is correct? | |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## Reference implementation

This is one defensible solution, not the only correct design. Identify at least one improvement before adopting it.

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import csv, json, logging

logger = logging.getLogger("phase1-lab")
if not logger.handlers:
    logger.addHandler(logging.StreamHandler())
logger.setLevel(logging.INFO)

@dataclass(frozen=True)
class LearnerRecord:
    learner_id: str
    name: str
    scores: tuple[float, ...]

    def __post_init__(self):
        if not self.learner_id.strip():
            raise ValueError("learner_id cannot be blank")
        if not self.name.strip():
            raise ValueError("name cannot be blank")
        if not self.scores:
            raise ValueError("at least one score is required")
        if any(not 0 <= score <= 100 for score in self.scores):
            raise ValueError("scores must be between 0 and 100")

    @property
    def average(self) -> float:
        return sum(self.scores) / len(self.scores)

raw = [
    {"learner_id":"L001","name":" Asha ","scores":[82,91,88]},
    {"learner_id":"L002","name":"Ravi","scores":[67,72,75]},
    {"learner_id":"L001","name":"Asha","scores":[82,91,88]},
    {"learner_id":"","name":"Invalid","scores":[110]},
]
accepted, rejected, seen = [], [], set()
for row in raw:
    try:
        record = LearnerRecord(
            learner_id=str(row["learner_id"]).strip(),
            name=str(row["name"]).strip(),
            scores=tuple(float(x) for x in row["scores"]),
        )
        if record.learner_id in seen:
            raise ValueError("duplicate learner_id")
        seen.add(record.learner_id)
        accepted.append(record)
    except (KeyError, TypeError, ValueError) as exc:
        rejected.append({"record":row,"reason":str(exc)})
        logger.warning("Rejected record: %s", exc)

summary = [
    {
        "learner_id":r.learner_id, "name":r.name, "score_count":len(r.scores),
        "average":round(r.average,2), "passed":r.average>=60,
    } for r in accepted
]
display(pd.DataFrame(summary))
display(pd.DataFrame(rejected))

def test_average():
    record = LearnerRecord("T1","Test",(50.0,100.0))
    assert record.average == 75.0
test_average()
print("Reference test passed.")

## Solution review

Review the reference under four lenses:

1. **Correctness:** Are contracts and calculations enforced?
2. **Robustness:** What failures remain unhandled?
3. **Maintainability:** Which responsibilities should become modules/functions?
4. **Decision validity:** Do outputs support the stated use without overclaiming?

Extend the implementation with one additional test and one observability improvement.